<a href="https://colab.research.google.com/github/hallockh/neur_265_spring2026/blob/main/notebooks/scRNAseq_03_03_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RNA-Sequencing: Single-Cell

DNA makes RNA makes Protein. This “Central Dogma” of molecular biology is a fundamental concept in molecular biology. With the development of DNA sequencing, we can explore the genome - all of the DNA in an organism. However, DNA is only the starting point. To understand what genes are active, and under what circumstances, we have to know what genes are being transcribed into messenger RNA. Starting from the first cell - in, for example, a human or a mouse - every other cell differentiates itself not by having different DNA, but by expressing (transcribing) those genes differently. A cell in the liver has the same DNA instructions as a neuron in the brain. However the genes being expressed differ greatly between these cells. The sum of all RNAs being expressed in a cell is known as the transcriptome.

##By the end of this notebook, you will be able to:

- Cluster single-cell RNA-sequencing (scRNA-seq) data by marker gene
- Map different "types" of cells onto each cluster
- Compare gene expression between clusters

##Introduction

RNA-Seq is a widely-used experiment that has become one of the most important ways to characterize the transcriptome. There are several variations, details, and statistics we are glossing over, but at its most basic we use an RNA-Seq experiment to ask the question - “does the gene expression in a control sample differ from the gene expression in an experimental sample?”

<img src = 'https://drive.google.com/uc?id=1ixFHqXm2_WBfwWqBIk0eAguqfafN3w-Y'>

##Steps for RNA-Sequencing

1. RNA is extracted from an organism of interest. Choices must be made at this stage about sampling of the organisms, since not all RNAs of interest may be present in every tissue and at all times. RNA is also easily degraded, and the extraction procedure can introduce biases into the collected sample. How we extract samples defines our experimental design. Typically we will sample at least two distinct groups (e.g. wild-type/mutant, healthy/disease, or control/experimental). We also will have several biological replicates since all our sampling will contain biases.

2. Several biochemical steps are taken to prepare extracted RNA into a sequencing library. RNA is converted into stable cDNA and undesired RNA (e.g. ribosomal RNA) is removed. Adaptors are attached to the cDNAs.

3. There are a variety of sequencing platforms that can generate the sequence of a prepared library. In the popular illumina protocol, millions of short cDNA fragments will be generated (usually 100-300bp in size).

4. Following sequencing, the RNA-Seq reads are quality controlled and then software is used to align and or assemble these reads for further analysis.

##Single-Cell RNA-Sequencing

One disadvantage to RNA-sequencing is that tissue is homogenized prior to RNA extraction. Brain tissue is heterogenous, containing many different types of cells (astrocytes, neurons, interneurons, *Pval* neurons, *Sst* neurons, etc.). Typical bulk RNA-sequencing strategies do not capture how the transcriptome might vary between these cell types. In order to analyze cell type-specific gene expression, we need to employ a technique called single-cell RNA-sequencing (scRNA-seq).

<img src = 'https://drive.google.com/uc?id=1QZVeic1CD3z6LK3GI_cDiBbE94GeeURk'>

In this technique, individual cells are identified based on presence of a specific marker gene, and then sorted into distinct compartments.

##Case Study Data

Today, we're going to look at a dataset of single-nucleus RNA-sequencing results from the entire human brain. The brains in this dataset came from three donors. The final dataset includes more than 3 million cells from 100 locations across the forebrain, midbrain, and hindbrain. The paper with these results can be found <a href="https://www.science.org/doi/10.1126/science.add7046">here</a>.

<img src="https://www.science.org/cms/10.1126/science.add7046/asset/54804481-ef05-40de-91f8-859a9ed8689e/assets/images/large/science.add7046-fa.jpg"/>

><b>Task:</b> First, import numpy, pandas, and matplotlib below:

In [ ]:
#Import numpy, pandas, and matplotlib


Next, import the data from the Allen Institute:

In [ ]:
!pip install "abc_atlas_access[notebooks] @ git+https://github.com/alleninstitute/abc_atlas_access.git" #Install the python library that we'll be using to access these data

from pathlib import Path

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

download_base = Path('../../data/abc_atlas') #Change directory to data download location
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

#Cluster annotation term sets

Each level of classification is represented as a <b><i>cluster annotation term set</b></i>. Each term set consists of a set of ordered terms. Each term set has a *label* (human readable string that is unique in the database), a *name*, *description* and *order* among the term sets.



In [ ]:
#View the term sets in this dataset

term_set = abc_cache.get_metadata_dataframe(directory='WHB-taxonomy', file_name='cluster_annotation_term_set')
term_set

# print out description for each term_set
for tsindex, tsrow in term_set.iterrows():
    print("%s:\n" % tsrow['name'])
    print("%s\n" % tsrow['description'])

#Cluster annotation terms

A <b><i>cluster annotation term</b></i> represents a group within a single level of classification. Each term object has a *label* (human readable string that is unique in the database), a *name*, which *cluster annotation term set* it belongs to and a *parent* or superclass if the term forms a hierarchy, an order within a *term set* and a *color* for visualization purposes.

In [ ]:
#Look at the first five clusters!

term = abc_cache.get_metadata_dataframe(directory='WHB-taxonomy', file_name='cluster_annotation_term', keep_default_na=False)
term.head(5)

><b>Task:</b> Look at the next 5 rows. In a new Markdown cell below, write a quick summary of what the values in the <code>name</code> column refer to. Are these clusters from a specific part of the brain? From a specific type of cell?

We can use the groupby and count functions in pandas to tally the number of terms in each term set (classification level).

In [ ]:
#Look at the number of each type of annotation term!

term[['label','cluster_annotation_term_set_name']].groupby('cluster_annotation_term_set_name').count()

><b>Task:</b> Find a way to visualize the above information in a pie chart in the code cell below!

In [ ]:
#Your pie chart here!


#Cluster to cluster annotation membership

The association between a <b><i>subcluster</b></i> and <b><i>cluster annotation term</b></i> is represented as a cluster to cluster annotation membership within the context of one cluster annotation term set. It is expected that a cluster in only associated with one term within a specific term set.

In [ ]:
# Count the number of clusters associated with each cluster annotation term
membership = abc_cache.get_metadata_dataframe(directory='WHB-taxonomy', file_name='cluster_to_cluster_annotation_membership')

term_cluster_count = membership.groupby(['cluster_annotation_term_label'])[['cluster_alias']].count()
term_cluster_count.columns = ['number_of_clusters']

term_by_label = term.set_index('label')
term_with_counts = term_by_label.join(term_cluster_count)
term_with_counts[['name', 'cluster_annotation_term_set_name', 'number_of_clusters', 'number_of_cells']].head(5)

Let’s visualize cluster and cells counts for of the classification levels using bar plots.

In [ ]:
def bar_plot_by_level_and_type(df, level, fig_width = 8.5, fig_height = 4):

    fig, ax = plt.subplots(1, 2)
    fig.set_size_inches(fig_width, fig_height)

    for idx, ctype in enumerate(['clusters', 'cells']):

        pred = (df['cluster_annotation_term_set_name'] == level)
        names = df[pred]['name']
        counts = df[pred]['number_of_%s' % ctype]
        colors = df[pred]['color_hex_triplet']

        ax[idx].barh(names, counts, color=colors)
        ax[idx].set_title('Number of %s by %s' % (ctype,level)),
        ax[idx].set_xscale('log')

        if idx > 0:
            ax[idx].set_yticklabels([])

    plt.show()

><b>Task:</b> Run the function that you created above, using your <code>term_with_counts</code> variable as your first input, and using the 'neurotransmitter' level as your second input.

In [ ]:
#Make a bar plot using your function!


><b>Question:</b> Which cells do you think are contained in the 'None' category for neurotransmitters?

><b>Task:</b> Re-create the bar plot above, but use supercluster as your category term, rather than neurotransmitter. Which supercluster contains the most neurons?

In [ ]:
#Make a bar plot with supercluster as the annotation term:


Finally, let's superimpose our neurotransmitter information onto each of our superclusters:

In [ ]:
#Make a pivot table with all clusters and term sets

pivot = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])['cluster_annotation_term_name'].first().unstack()
pivot = pivot[term_set['name']] # order columns
pivot

color = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])['color_hex_triplet'].first().unstack().fillna('#f9f9f9')
color = color[term_set['name']] # order columns
color.columns = ['%s_color' % x for x in color.columns]
color

#Make some functions for creating cluster count tables and stacked bar plots

def distribution(A, B):

    AxB = pivot.groupby([A, B])[['cluster']].count()
    AxB.columns = ['number_of_clusters']
    AxB = AxB.unstack().fillna(0)

    B_names = [x[1] for x in list(AxB.columns)]
    pred = (term['cluster_annotation_term_set_name'] == B)
    term_by_name = term[pred].set_index('name')
    B_colors = term_by_name.loc[B_names, 'color_hex_triplet']

    return AxB, B_names, B_colors

def stacked_bar_distribution(AxB, B_names, B_colors, fig_width = 6, fig_height = 6):

    fig, ax = plt.subplots()
    fig.set_size_inches(fig_width, fig_height)

    bottom = np.zeros(len(AxB))

    for i, col in enumerate(AxB.columns):
        ax.barh(AxB.index, AxB[col], left=bottom, label=col[1], color=B_colors[i])
        bottom += np.array(AxB[col])

    ax.set_title('Distribution of %s in each %s' % (AxB.columns.names[1], AxB.index.name))
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

    plt.show()

    return fig, ax

#Make a stacked bar plot

AxB, B_names, B_colors = distribution('supercluster', 'neurotransmitter')
fig, ax = stacked_bar_distribution(AxB, B_names, B_colors, 6, 6)
plt.show()



><b>Task:</b> In a new Markdown cell below, describe what you see in your stacked bar plot. What can you say about the relationship between neurotransmitter synthesis and brain region? How about neurotransmitter synthesis and cell type? Do brain regions and/or cell types tend to produce specific neurotransmitters? Does neurotransmitter identity seem to be parcellated in the brain, or does it seem to be widespread?

###Your stacked bar plot interpretation here!

#UMAP spatial embedding

Now that we’ve merged the cluster metadata into the main cells data, we can plot the Uniform Manifold Approximation and Projection (UMAP) for all the cells in the dataset using information from the clusters. The UMAP is a dimension reduction technique that can be used for visualizing and exploring large-dimension datasets. The x, y columns of the cell metadata table represents the coordinate of the all cells UMAP in Figure 1B of the manuscript. Note that the (x, y) coordinates for Neuron and Non-neuron cells overlap and should be plotted seperately.

We define a small helper function <code>plot umap</code> to visualize the cells on the UMAP. In this example will will plot associated cell information colorized by: dissection region of interest, neurotransmitter identity, cell supercluster, cluster, and subcluster. For ease of demostration, we do a simple subsampling of the cells by a factor of 10 to reduce processing time.

><b>Task:</b> What is a UMAP? Do some research on your own by reading through this <a href="https://alleninstitute.org/resource/what-is-a-umap/">Allen Institute resource</a>, and briefly write your understanding of what this technique is in a new Markdown cell below:

###Give your UMAP summary here!

In [ ]:
#Get our experiment metadata (names and labels for everything)
cell = abc_cache.get_metadata_dataframe(
    directory='WHB-10Xv3',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))

#Sort our data by membership category
membership = abc_cache.get_metadata_dataframe(
    directory='WHB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership'
)
membership_groupby = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])

term_sets = abc_cache.get_metadata_dataframe(directory='WHB-taxonomy', file_name='cluster_annotation_term_set').set_index('label')
cluster_details = membership_groupby['cluster_annotation_term_name'].first().unstack()
cluster_details = cluster_details[term_sets['name']] # order columns
cluster_details.fillna('Other', inplace=True)
cluster_details.sort_values(['supercluster', 'cluster', 'subcluster'], inplace=True)
cluster_details.head(5)

#Set some asthetic properties for our UMAP
cluster_colors = membership_groupby['color_hex_triplet'].first().unstack()
cluster_colors = cluster_colors[term_sets['name']]
cluster_colors.sort_values(['supercluster', 'cluster', 'subcluster'], inplace=True)

roi = abc_cache.get_metadata_dataframe(directory='WHB-10Xv3', file_name='region_of_interest_structure_map')
roi.set_index('region_of_interest_label', inplace=True)
roi.rename(columns={'color_hex_triplet': 'region_of_interest_color'},
           inplace=True)

cell_extended = cell.join(cluster_details, on='cluster_alias')
cell_extended = cell_extended.join(cluster_colors, on='cluster_alias', rsuffix='_color')
cell_extended = cell_extended.join(roi[['region_of_interest_color']], on='region_of_interest_label')


In [ ]:
#Make a UMAP plotting function
def plot_umap(xx, yy, cc=None, val=None, fig_width=8, fig_height=8, cmap=None):

    fig, ax = plt.subplots()
    fig.set_size_inches(fig_width, fig_height)

    if cmap is not None:
        plt.scatter(xx, yy, s=0.5, c=val, marker='.', cmap=cmap)
    elif cc is not None:
        plt.scatter(xx, yy, s=0.5, color=cc, marker='.')

    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    return fig, ax

In [ ]:
#We have to dissociate neurons from non-neurons - otherwise they would overlap
neurons_subsampled = cell_extended[cell_extended['feature_matrix_label'] == 'WHB-10Xv3-Neurons'][::10]
non_neurons_subsampled = cell_extended[cell_extended['feature_matrix_label'] == 'WHB-10Xv3-Nonneurons'][::10]

#Make a UMAP!
fig, ax = plot_umap(neurons_subsampled['x'], neurons_subsampled['y'], cc=neurons_subsampled['region_of_interest_color'])
res = ax.set_title("Neurons: Dissection Region Of Interest")
plt.show()
fig, ax = plot_umap(non_neurons_subsampled['x'], non_neurons_subsampled['y'], cc=non_neurons_subsampled['region_of_interest_color'])
res = ax.set_title("Non-neurons: Dissection Region Of Interest")
plt.show()

><b>Task:</b> Interpret your UMAP in a Markdown cell below! How well do the genetic identities of neurons map onto the brain region that they were dissected from? How about non-neurons?

###Your UMAP interpretation here!

><b>Task:</b> Make another UMAP plot, but this time with neurotransmitter expression mapped onto the different superclusters. How well does neurotransmitter identity map onto genetic identity for neurons? How about non-neurons?

In [ ]:
#Your neurotransmitter UMAP here!


#Single cell transcriptomes

The ~3 million cell dataset of WHB has been divided into 2 expression matrices: one with Neurons and one with Nonneurons. Each matrix file is formatted as an annadata, h5ad file with minimal metadata.

What are the major genes that are driving differentiation of these clusters? Let's look at expression of some specific genes to find out!

In this section, we show a use case with the example genes *SLC17A6, SLC17A7, SLC32A1, PTPRC, PLP1, AQP4*, and *TTR*. These genes were selected because they were presented in Siletti et al. 2023 as markers genes for glutamatergic (*SLC17A6, SLC17A7*) and GABAergic (*SLC32A1*) neurons, immune cells (*PTPRC*), oligodendrocytes (*PLP1*), astrocytes (*AQP4*), and choroid plexus (*TTR*). “Marker genes” have much higher expression in the specified cell type or anatomic structure when compared to all other cells, and in many cases are functionally relevant for those cell types.

In [ ]:
#Grab information and metadata about each individual gene in the dataset

gene = abc_cache.get_metadata_dataframe(directory='WHB-10Xv3', file_name='gene')
gene.set_index('gene_identifier', inplace=True)
print("Number of genes = ", len(gene))

example_cells_with_genes = abc_cache.get_metadata_dataframe(
    directory='WHB-10Xv3',
    file_name='example_genes_all_cells_expression'
).set_index('cell_label')

cell_extended_with_genes = cell_extended.join(example_cells_with_genes)

In [ ]:
#Function to compute average expression by category

def aggregate_by_metadata(df, gnames, value, sort = False):
    grouped = df.groupby(value)[gnames].mean()
    if sort:
        grouped = grouped.sort_values(by=gnames[0], ascending=False)
    return grouped

In [ ]:
#Function to plot a heatmap!

def plot_heatmap(df, fig_width=8, fig_height=4, cmap=plt.cm.magma_r) :

    arr = df.to_numpy()

    fig, ax = plt.subplots()
    fig.set_size_inches(fig_width, fig_height)

    im = ax.imshow(arr, cmap=cmap, aspect='auto', vmin=0, vmax=6)
    xlabs = df.columns.values
    ylabs = df.index.values

    ax.set_xticks(range(len(xlabs)))
    ax.set_xticklabels(xlabs)

    ax.set_yticks(range(len(ylabs)))
    res = ax.set_yticklabels(ylabs)

    return im

In [ ]:
#Plot our heatmap!

agg = aggregate_by_metadata(cell_extended_with_genes, example_cells_with_genes.columns, 'supercluster')
res = plot_heatmap(agg, 8, 8)
plt.show()

How about mapping individual genes onto our UMAP? Is a *SLC17A6* a good marker gene for this dataset?

In [ ]:
#Map an individual gene onto our UMAP plot

neurons_subsampled = cell_extended_with_genes[cell_extended_with_genes['feature_matrix_label'] == 'WHB-10Xv3-Neurons'][::10]
non_neurons_subsampled = cell_extended_with_genes[cell_extended_with_genes['feature_matrix_label'] == 'WHB-10Xv3-Nonneurons'][::10]

fig, ax = plot_umap(neurons_subsampled['x'], neurons_subsampled['y'], val=neurons_subsampled['SLC17A6'], cmap=plt.cm.magma_r)
res = ax.set_title("Neuron: Gene SLC17A6")
plt.show()
fig, ax = plot_umap(non_neurons_subsampled['x'], non_neurons_subsampled['y'], val=non_neurons_subsampled['SLC17A6'], cmap=plt.cm.magma_r)
res = ax.set_title("Non-Neuron: Gene SLC17A6")
plt.show()

><b>Task:</b> Map the other marker genes in the subsampled dataset onto your UMAP plot. Which one(s) do you think best differentiate cells in the different superclusters?

###Your marker gene explanation here!